# 02. Preprocesamiento de Texto

Tokenización, lematización y limpieza lingüística con spaCy.


In [ ]:
import pandas as pd
import re
import unicodedata
import spacy
from nltk.corpus import stopwords
from pathlib import Path
from tqdm.notebook import tqdm

DATA_INTERIM = Path("data/interim")
df = pd.read_csv(DATA_INTERIM / "01_limpio.csv")
print(f"Cargado: {len(df):,} filas")


In [ ]:
# Configurar spaCy y stopwords
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
additional_stops = {"xxxx", "xx", "xxxxx", "xx/xx/xxxx", "xxx", "company", "consumer", "complaint", "report", "account", "information", "requested", "please", "also", "would", "could", "should", "said", "told", "called", "spoke", "stated", "mentioned", "however", "therefore", "furthermore", "accordingly"}
stop_words = set(stopwords.words("english")) | additional_stops

def clean_text(text):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\b[Xx]{2,}\b", "[MASK]", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def lemmatize(text):
    doc = nlp(text)
    return [token.lemma_.lower().strip() for token in doc if not token.is_space and not token.is_punct and not token.is_digit and token.lemma_.lower().strip() not in stop_words and len(token.lemma_.lower().strip()) > 1]


In [ ]:
# Aplicar a muestra (ajustar SAMPLE_SIZE según recursos)
SAMPLE_SIZE = 5000
df_sample = df.head(SAMPLE_SIZE).copy()

cleaned = [clean_text(t) for t in df_sample["Consumer complaint narrative"].astype(str)]
tokens = []
for doc in tqdm(nlp.pipe(cleaned, batch_size=500), total=len(cleaned)):
    tokens.append([token.lemma_.lower().strip() for token in doc if not token.is_space and not token.is_punct and not token.is_digit and token.lemma_.lower().strip() not in stop_words and len(token.lemma_.lower().strip()) > 1])

df_sample["tokens"] = tokens
df_sample["processed_text"] = df_sample["tokens"].apply(lambda x: " ".join(x))
df_sample.to_csv(DATA_INTERIM / "02_preprocesado.csv", index=False)
print("Preprocesamiento completado.")
